DEPLOY DI MODELLI AI CON DOCKER

Quando il modello di deep learning funziona sul computer, ma sul server del cliente o sul cloud tutto smette di funzionare.

Immagina di non dover spedire solo il codice, ma di poter spedire l'intero computer configurato come vuoi, dentro una scatola di acciaio virtuale
Questo è il potere della containerizzazione

- Containerizzazione: creare un Dockerfile per isolare codice e driver GPU
- Ottimizzazione: nel mondo dell'ai le immagini diventa giganti e pesanti, è possibile farle dimagrire senza perdere potenza, immagini 'slim' e multi-stage builds per ridurre il footprint
- Riproducibilità: allineamento tra ambienti di sviluppo e produzione. Come garantire che la nostra pipeline sia una roccia di stabilità identica oggi come tra un anno.

Perchè l'ai ha cosi bisogno di isolamento?
Un modello di Deep Learning non è solo codice Python è un ecosistema delicatissimo.
I modelli di Deep Learning dipendono da versioni specifiche di Python librerie (PyTorch, TensorFlow) e drivere di basso livello, Docker permete di impacchettare tutto questo in un'unità atomica chiamata immagine.
Dockerfile include on solo il codice della nostra API FastAPI, ma anche il supporto hardware necessario per l'inferenza accelerata.
Una volta fatta la scatola è immutabile, se la porti un un server in America o in Giappone, il risultato del calcolo sarà identico.

Per creare questo calco abbiamo bisogno di una ricetta precisa, il DockerFile

Il dockerfile è come una ricetta di cucina

Le istruzioni per la costruzione
- Istruzione FROM: scelta dell'immagine di base, tipicamente un'immagine NVIDIA CUDA per garantire l'accesso ai core della GPU.
- Istruzione RUN: esecuzione di comandi per installare dipendenze di sistma e paccheetti Python definiti in 'requirements.txt'
- Istruzioni COPY: trasferimento del codice sorgente e dei pesi del modello locale all'interno del file system isolato del container. Portiamo il codice ed i pesi dentro il contenitore.
- L'immagine finale è composta da una serie di layer immutabili che garantiscono la persistenza dell'ambiente.

Ogni comando crea un layer, se cambiate solo il codice ma non le librerie, docker non riscasrica tutto, ma riutilizza i pezzi precedenti non modificati

Ma come facciamo a far capire a questa scatola che deve usare la potenza della nostra scheda video?
Qui entra in gioco NVIDIA Container Toolkit
Accesso centralizzato agli standard output del modello per monitorare errori di inferenza a runtime.
Le Envirinment Variables sono istruzioni Docker per monitorare se l'API AI è ancora reattiva o se il processo è andato in deadlock
Gli Entrypoint utilizzano mappe locali per analizzare dati di input o salvare log persistenti fuori dal container effimero.

Piccolo segreto per velocizzare il lavoro quotidiano
Gerarchia dei Layer
Ottimizzazione della cache
Docker riutilizza i layer non modificati per velocizzare le build successive. E' fondamentale ordinare le istruzioni partendo da quelle che cambiano meno spesso.
La velocità di deploy dipende dalla capacità di minimizzare il numero di layer ricostruiti ad ogni modifica del codice.

Ottimizzazione delle Immagini
Ridurre il peso delle pipeline AI
Le immagini AI possono facilmente superare 10 GB, tendono a diventare enormi. Questo rallenta il trasferimento sui server cloud e aumenta i costi di storage oltre al tempo.
Esistono tecniche avanzate per asciugare l'immagine finale, mantenendo solo l'essenziale per l'esecuzione del modello.

Quali tecniche usiamo per far dimagrire i nostri container

La tecnica regina è il multi-stage
- Immagini Slim: uso di versioni ridotte di Debian o Ubuntu per eliminare compilatori e strumenti inutili in fase di runtime
- Multi-stage Build: tecnica che usa un'immagine pesante per compilare le dipendenze (compilare e installare tutto) e una leggera per l'esecuzione finale.
- Layer Flattening: minimizzare il numero di istruzioni RUN per ridurre l'overhead del file system di Docker
- Il rapporto di compressione tra un'immagine standard e una ottimizzata può superare il 60%

Oltre al multi-stage ci sono piccoli gesti quotidiani di pulizia da non dimenticare

Rimozione dei Residui
Dobbiamo essere dei chirurghi della pulizia
- Utilizzo del falg '--no_cache-dir' durante l'installazione dei pacchetti per evitare di salvare file temporanei inutili (Cleanup di Pip)
- Esclusione di file pesanti non necessari (es. dataset di training, cartelle .git, cache locali) dalla build dell'immagine (dockerignore)
- Considerare il caricamento dei pesi del modello tramite volumi esterni o cloud storage anzichè includerli direttamente nell'immagine (external model storage)

Ma perchè tutta questa osessione per il megabyte?

Analisi dello Storage
Costi di Distribuzione
Nel cloud il tempo è denaro.
Un'immagine più piccola non è solo più veloce, ma anche più sicura, poichè riduce la superficie di attacco eliminando binari non necessari.
La latenza di pull in un cluster Kubernetes è direttamente proporzionale alla dimensione dell'immagine.
Un immagine ottimizzata significa un app che reagisce istantaneamente al carico, rendendo il servizio ai elastico ed efficiente

Stabilità nel tempo
Riproducibilità Garantita
L'ambiente come codice
Il problema del 'funziona sul mio pc' sparisce con Dockeer.
L'ambiente di produzione diventa una copia esatta dell'ambiente di test.
E' necessario bloccare le versioni (pinning) per evitare che aggiornamenti imprevisti rompano la nostra pipeline AI mesi dopo il deploy.

E' necessario bloccare questo stato di grazie tramite il pinning
Mai usare l'ultima versione di Python
Stabilizzare la pipeline
- Version Pinning: definire versioni esatte per l'immagine base e per ogni libreria Python (es. Torch==2.1.0)
- Immutable Tags: evitare l'uso di tag 'latest' per le immagini prodotte, preferendo hash o versione semantiche (v.1.0.1)
- Deterministic Build: garantire che ricreare l'immagine produca sempre lo stesso output binario
- La riproducibilità è l'uguaglianza tra lo stato dell'ambiente di sviluppo e quello di produzione.
L'obbiettivo è uguaglianza assoluta da sviluppo e produzione

Monitoraggio e Debug
- Container Logs: accesso centralizzato agli standard output del modello per monitorare errori di inferenza a runtime
- Health Checks: istruzioni Docker per monitorare se l'API AI è ancora reattiva o se il processo è andato in deadlock
- Valume Mapping: utilizzo di mappe locali per analizzare dati di input o salvare log persistenti fuori dal container effimero.

Questo ci permette di integrare l'AI in flusso di lavoro completamente automatizzati

Continuous Deployment
Integrazione nei workflow
Docker è il tassello fondamentale delle pipeline di CI/CD moderne, permettendo test automatizzati in ambienti isolati
L'automazione della build riduce l'errore umano nel setup dei server remoti.
In una pipeline moderna, si carica il codice su github e un sistema automatico costuisce l'immagine e li lancia sul server.
Automatizzando tutto, riduciamo la probabilità di errore umano.




